<a href="https://colab.research.google.com/github/rameshjayamani-oss/GenAI_Assignment/blob/main/Assignment_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-community
!pip install -q pymupdf
!pip install -q langchain-google-genai
!pip install -q chromadb
!pip install -q langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━

In [2]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_831/2747336630.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


In [3]:
os.environ["GOOGLE_API_KEY"] = userdata.get('apikey')
policy_file_url="https://customer-portal-assets.hdfcergo.com/documents/OptimaPlus-192946032259.pdf"
# policy_file_url = "https://github.com/MalavMDesai/GenAIAssignment/raw/refs/heads/main/Assignment5_Policy-1-5.pdf"

loader = PyMuPDFLoader(policy_file_url)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)


In [4]:
print(len(chunks))
print(chunks[0])

84
page_content='OPTIMA PLUS - Prospectus 
HDFC ERGO General Insurance Limited 
 
HDFC ERGO General Insurance Company Limited. IRDAI Reg. No.146. CIN: U66030MH2007PLC177117. Registered & 
Corporate Office: 6th Floor, Leela Business Park, Andheri-Kurla Road, Andheri (East), Mumbai – 400 059. UIN: 
Optima Plus - HDHHLIP21336V022021  
 
1 
Optima Plus - Prospectus 
Eligibility 
▪ 
This policy covers persons in the age group 91 days to 65 years.  
▪ 
The maximum entry age is restricted upto 65 years. 
▪ 
Child between 91 days to 5 years can be insured only when either parent is getting 
insured under this policy.  
▪ 
The policy offers coverage on individual sum insured basis.  
▪ 
This policy can be issued to an individual and/or family. 
▪' metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-05-06T10:28:39+05:30', 'source': 'https://customer-portal-assets.hdfcergo.com/documents/OptimaPlus-192946032259.pdf', 'fil

In [5]:
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", task_type="retrieval_document")
vector_store = Chroma.from_documents(chunks, embedding)
retriever = vector_store.as_retriever()

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0.0)#, override_model_name=True)

system_prompt = (
    "You work as a expert Insurance Claims agent in HDFC ERGO General Insurance Company Limited \n"
    "Answer the questions using only provided policy context. \n"
    "If you do not know the answer or if it is not is the provided document than exactly say:\n"
    "'Unable to find the information in policy document, connect to customer care'"
    "CONTEXT:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', "{input}")
])

def format_docs(docs):
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "input": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# query1 = "What is the waiting period for pre-existing diseases?"
# res = rag_chain.invoke(query1)

# print(f"User query: {query1}")
# print(f"ans: \n {res}")


In [9]:
policy_queries = [
    "What is the entry age limit?",
    "Is maternity covered?",
    "What is the waiting period for pre-existing diseases?",
    "What documents are required for filing a claim?",
    "what is not covered?",
]
for index, query in enumerate(policy_queries, start=1):
  res = rag_chain.invoke(query)
  print(f"Processing Query #{index}: {query}")
  print(f"Rag Answers #{index}: \n {res} \n")

Processing Query #1: What is the entry age limit?
Rag Answers #1: 
 Based on the provided policy document, the entry age limits are as follows:

* The policy covers persons in the age group of **91 days to 65 years**.
* The **maximum entry age** is restricted up to **65 years**.
* A child between **91 days to 5 years** can be insured only when either parent is also insured under this policy. 

Processing Query #2: Is maternity covered?
Rag Answers #2: 
 Based on the provided policy context, maternity is excluded from coverage (under Code – Excl18). This exclusion includes:

* Medical treatment expenses traceable to childbirth (including complicated deliveries and caesarean sections incurred during hospitalization), except for ectopic pregnancy.
* Expenses towards miscarriage (unless due to an accident) and lawful medical termination of pregnancy during the Policy period. 

Processing Query #3: What is the waiting period for pre-existing diseases?
Rag Answers #3: 
 Based on the provided